# Preprocessing Data & Model Baseline Deteksi Kerusakan iPhone

In [ ]:
import pandas as pd
import numpy as np
import os
import re
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

print("Pandas version:", pd.__version__)
print("NumPy version:", np.__version__)

### 1. Loading Data Awal

In [ ]:
data_dir = "data"
toko_path = os.path.join(data_dir, "data toko 2.xlsx")
utama_path = os.path.join(data_dir, "data utama.xlsx")

# Membaca data toko (Sheet JAN dan FEB)
df_jan = pd.read_excel(toko_path, sheet_name="JAN")
df_feb = pd.read_excel(toko_path, sheet_name="FEB")
df_toko = pd.concat([df_jan, df_feb], ignore_index=True)

# Membaca data utama (Tanpa header, index baris 0 akan diabaikan jika berisi header)
df_utama = pd.read_excel(utama_path, header=None)

print(f"Data Toko 2 (JAN + FEB) - Jumlah baris: {df_toko.shape[0]}, Jumlah kolom: {df_toko.shape[1]}")
print(f"Data Utama - Jumlah baris: {df_utama.shape[0]}, Jumlah kolom: {df_utama.shape[1]}")

### 2. Normalisasi Teks (Lexicon Normalization) & Cleaning
Pada tahap ini kita mendefinisikan kamus normalisasi istilah non-baku (slang/typo) khas Indonesia untuk menstandarkan catatan teknisi sebelum ekstraksi fitur dilakukan. Ini membuat data latih menjadi konsisten dan meningkatkan akurasi klasifikasi model.

In [ ]:
def normalize_text(text):
    text = str(text).lower().strip()
    
    # Kamus pemetaan slang/typo bahasa Indonesia ke kata baku
    slang_map = {
        r'\bbatrai\b': 'baterai',
        r'\bbatre\b': 'baterai',
        r'\bbatrey\b': 'baterai',
        r'\bbattery\b': 'baterai',
        r'\bbatarai\b': 'baterai',
        r'\bgembung\b': 'kembung',
        r'\blembung\b': 'kembung',
        r'\bgakbisa\b': 'tidak bisa',
        r'\bgkbisa\b': 'tidak bisa',
        r'\bgabisa\b': 'tidak bisa',
        r'\bgak bisa\b': 'tidak bisa',
        r'\bgk bisa\b': 'tidak bisa',
        r'\bga bisa\b': 'tidak bisa',
        r'\bkoncas\b': 'konektor cas',
        r'\bkon cas\b': 'konektor cas',
        r'\blubang cas\b': 'konektor cas',
        r'\bport cas\b': 'konektor cas',
        r'\bport charger\b': 'konektor cas',
        r'\bmatot\b': 'mati total',
        r'\bshort\b': 'korsleting',
        r'\bkonslet\b': 'korsleting',
        r'\bblkg\b': 'belakang',
        r'\bdicharge\b': 'dicas',
        r'\bdi charge\b': 'dicas',
        r'\bflexible\b': 'fleksibel',
        r'\bfleksi\b': 'fleksibel',
        r'\bflexy\b': 'fleksibel',
        r'\bflexyble\b': 'fleksibel',
        r'\bfelxyble\b': 'fleksibel',
        r'\bts\b': 'touchscreen',
        r'\blayar\b': 'lcd',
        r'\bscreen\b': 'lcd',
        r'\bcamera\b': 'kamera',
        r'\bbackdor\b': 'backdoor',
        r'\bbckdor\b': 'backdoor',
        r'\bback glass\b': 'backdoor',
        r'\bbackglass\b': 'backdoor',
        r'\bkaca belakang\b': 'backdoor',
        r'\bpunggung\b': 'backdoor',
        r'\bon off\b': 'on/off',
        r'\bonoff\b': 'on/off',
        r'\bbutton\b': 'tombol',
        r'\bbtn\b': 'tombol',
        r'\bcharge\b': 'cas',
        r'\bcharger\b': 'cas',
        r'\bathena\b': 'antena'
    }
    
    for pattern, replacement in slang_map.items():
        text = re.sub(pattern, replacement, text)
        
    # Bersihkan spasi berlebih
    text = re.sub(r'\s+', ' ', text).strip()
    return text

records = []

# Pembersihan untuk data toko 2
for idx, row in df_toko.iterrows():
    hp = str(row['TIPE HP']).upper()
    if 'IPHONE' not in hp:
        continue
    
    kerusakan = str(row['KERUSAKAN']).lower()
    
    # Filter masalah software
    sw_keywords = ['lupa pola', 'flash', 'sofware', 'software', 'instal ulang', 'icloud', 'cleaning', 'stuck logo', 'bypass']
    if any(k in kerusakan for k in sw_keywords):
        continue
    
    # Hilangkan action verbs dari Symptom_Text untuk mengurangi target leakage pada Toko 2
    symptom_cleaned = kerusakan
    for verb in ['ganti ', 'servis ', 'lem ulang ', 'lem ', 'bersihkan ', 'cek ']:
        symptom_cleaned = symptom_cleaned.replace(verb, '')
        
    records.append({
        'Source': 'Toko 2',
        'Tipe HP': hp,
        'Symptom_Text': normalize_text(symptom_cleaned),
        'Context_Text': '',
        'Action_Text': normalize_text(kerusakan)
    })

# Pembersihan untuk data utama
for idx, row in df_utama.iterrows():
    if idx == 0 and "Wahid Willy" in str(row[1]):
        continue
    hp = str(row[2]).upper()
    if 'IPHONE' not in hp:
        continue
        
    s1 = str(row[4]).lower()
    s2 = str(row[5]).lower()
    action = str(row[6]).lower()
    
    # Filter masalah software
    sw_keywords = ['lupa pola', 'flash', 'sofware', 'software', 'instal ulang', 'icloud', 'cleaning', 'stuck logo', 'bypass', 'downgrade']
    if any(k in s1 for k in sw_keywords) or any(k in s2 for k in sw_keywords) or any(k in action for k in sw_keywords):
        continue
        
    records.append({
        'Source': 'Utama',
        'Tipe HP': hp,
        'Symptom_Text': normalize_text(s1),
        'Context_Text': normalize_text(s2),
        'Action_Text': normalize_text(action)
    })

df_raw = pd.DataFrame(records)
print(f"Jumlah total record hardware setelah pembersihan & normalisasi awal: {len(df_raw)}")

### 3. Ekstraksi Fitur & Mapping ke Fitur Gejala Terstruktur


In [ ]:
from IPython.core import release
def match_keywords(text, keywords):
    patterns = []
    for k in keywords:
        escaped = re.escape(k).replace(r'\ ', r'\s+')
        patterns.append(rf'\b{escaped}\b')
    pattern = '|'.join(patterns)
    return bool(re.search(pattern, text, re.IGNORECASE))

def map_row(row):
    # Fitur gejala HANYA menggunakan Symptom + Context untuk menghindari Target Leakage
    sym_text = str(row['Symptom_Text']).lower() + " " + str(row['Context_Text']).lower()
    act_text = str(row['Action_Text']).lower()
    
    # 1. Signal
    if match_keywords(sym_text, ['sinyal hilang', 'signal hilang', 'hilang jaringan', 'modem', 'baseband', 'no signal', 'keblokir']):
        signal = 'Hilang Total'
    elif match_keywords(sym_text, ['sinyal lemah', 'signal lemah', 'sinyal naik turun', 'signal hilang timbul']):
        signal = 'Bar Rendah'
    else:
        signal = 'Normal'
        
    # 2. Baterai
    if match_keywords(sym_text, ['baterai', 'kembung', 'drop', 'boros', 'mentok 0%', 'turun']):
        baterai = 'Daya Tahan Rendah/Gembung'
    else:
        baterai = 'Normal'
        
    # 3. LCD
    has_lcd_kw = match_keywords(sym_text, ['greenscreen', 'greenline', 'hijau'])
    has_gelap_kw = match_keywords(sym_text, ['blank', 'blackscreen', 'gelap', 'retak', 'garis', 'pecah']) and not match_keywords(sym_text, ['backdoor', 'kaca belakang', 'punggung', 'housing', 'casing', 'bodi'])
    
    # Konteks pecah (bukan di speaker atau backdoor)
    has_pecah_general = match_keywords(sym_text, ['pecah']) and not match_keywords(sym_text, ['suara', 'speaker', 'audio', 'buzzer', 'backdoor', 'kaca belakang', 'punggung', 'housing', 'casing', 'bodi'])
    
    if has_lcd_kw:
        lcd = 'Greenscreen'
    elif has_gelap_kw or has_pecah_general:
        lcd = 'Gelap'
    else:
        lcd = 'Normal'
        
    # 4. Wifi
    if match_keywords(sym_text, ['wifi']):
        wifi = 'Hilang Timbul'
    else:
        wifi = 'Normal'
        
    # 5. Touchscreen
    if match_keywords(sym_text, ['disentuh', 'sentuh', 'touchscreen', 'ghostouch', 'ngefreeze']):
        touch = 'Tidak bisa disentuh'
    else:
        touch = 'Normal'
        
    # 6. Tegangan
    if match_keywords(sym_text, ['korsleting', 'mati total', 'mati', 'korosi', 'basah', 'kena air', 'masuk air']):
        if match_keywords(sym_text, ['tombol', 'kamera', 'speaker', 'wifi']):
            tegangan = '0,8 - 2,2'
        else:
            tegangan = '0,0'
    elif match_keywords(sym_text, ['tegangan rendah', 'vcc main', 'baterai drop']):
        tegangan = '0,6'
    else:
        tegangan = '0,8 - 2,2'
        
    # 7. Kamera
    if match_keywords(sym_text, ['kamera', 'foto', 'buram', 'jamur', 'getar']):
        kamera = 'Blank'
    else:
        kamera = 'Normal'
        
    # 8. Konektor Cas
    if match_keywords(sym_text, ['tidak bisa dicas', 'dicas mati', 'dicas tidak naik', 'dicas keluar masuk', 'konektor cas', 'dicas', 'cas', 'konektor']):
        if match_keywords(sym_text, ['keluar masuk', 'longgar', 'kotor']):
            konektor = 'Keluar masuk'
        else:
            konektor = 'Tidak Ngecas'
    else:
        konektor = 'Normal'
        
    # 9. Speaker
    has_speaker_kw = match_keywords(sym_text, ['suara', 'speaker', 'buzzer', 'audio', 'mikrofon', 'microphone', 'tidak terdengar', 'rekam suara'])
    has_pecah_speaker = match_keywords(sym_text, ['pecah']) and match_keywords(sym_text, ['suara', 'speaker', 'buzzer', 'audio'])
    
    if has_speaker_kw or has_pecah_speaker:
        speaker = 'rusak'
    else:
        speaker = 'Normal'
        
    # 10. Backdoor
    if match_keywords(sym_text, ['backdoor', 'kaca belakang', 'punggung', 'housing', 'casing', 'bodi']):
        backdoor = 'Pecah/Retak'
    else:
        backdoor = 'Normal'
        
    # 11. Tombol
    if match_keywords(sym_text, ['tombol', 'power', 'volume', 'haptic', 'haptik', 'switch', 'silent', 'home', 'on off', 'on/off']):
        tombol = 'Tidak Berfungsi'
    else:
        tombol = 'Normal'
        
    # Target Label: Kerusakan (13 Kategori)
    damage = None
    if match_keywords(act_text, ['baterai', 'konektor baterai', 'soket baterai', 'fuse']):
        damage = 'Baterai Rusak'
    elif match_keywords(act_text, ['lcd', 'touchscreen', 'glass', 'sentuh', 'pindah chip', 'jalur lcd', 'lampu', 'display', 'screen', 'greenscreen', 'whitescreen', 'blackscreen', 'greenline']):
        damage = 'LCD Rusak'
    elif match_keywords(act_text, ['speaker', 'buzzer', 'earpiece', 'suara pecah']):
        damage = 'Speaker Rusak'
    elif match_keywords(act_text, ['ic power', 'vcc main', 'cpu', 'power', 'jalur vcc', 'mesin', 'jalur', 'short', 'korsleting', 'mati', 'mati total', 'mati cek', 'korosi', 'swap board']):
        damage = 'IC Power Rusak'
    elif match_keywords(act_text, ['ic cas', 'ic charger', 'ic charging', 'cas rusak', 'hydra']):
        damage = 'IC Cas Rusak'
    elif match_keywords(act_text, ['baseband', 'ic rf', 'ic signal', 'ic wtr', 'wtr', 'ic rf/baseband']):
        damage = 'IC WTR Rusak'
    elif match_keywords(act_text, ['antena sinyal', 'antena signal', 'fleksibel antena', 'sinyal', 'signal']):
        damage = 'Antena Signal Rusak'
    elif match_keywords(act_text, ['kamera', 'lens', 'flash', 'kamera depan', 'kamera belakang']):
        damage = 'Kamera Rusak'
    elif match_keywords(act_text, ['mikrofon', 'mic', 'microfon']):
        damage = 'Mikrofon Rusak'
    elif match_keywords(act_text, ['konektor cas', 'port cas', 'fleksibel cas', 'charger kotor', 'cas', 'konektor']):
        damage = 'Port Pengisian Rusak'
    elif match_keywords(act_text, ['ic audio', 'audio', 'audio rusak']):
        damage = 'IC Audio Rusak'
    elif match_keywords(act_text, ['backdoor', 'kaca belakang', 'punggung', 'housing', 'casing', 'bodi']):
        damage = 'Backdoor Rusak'
    elif match_keywords(act_text, ['tombol', 'power', 'volume', 'haptic', 'haptik', 'switch', 'silent', 'home', 'on off', 'on/off']):
        damage = 'Tombol Rusak'
        
    if damage is None and row['Source'] == 'Toko 2':
        # Fallback khusus Toko 2 karena kolom data tunggal
        if match_keywords(sym_text, ['baterai', 'soket baterai', 'konektor baterai', 'fuse']):
            damage = 'Baterai Rusak'
        elif match_keywords(sym_text, ['lcd', 'touchscreen', 'glass', 'sentuh', 'pindah chip', 'jalur lcd', 'lampu', 'display', 'screen', 'greenscreen', 'whitescreen', 'blackscreen', 'greenline']):
            damage = 'LCD Rusak'
        elif match_keywords(sym_text, ['speaker', 'buzzer', 'earpiece', 'suara pecah']):
            damage = 'Speaker Rusak'
        elif match_keywords(sym_text, ['ic power', 'vcc main', 'cpu', 'power', 'jalur vcc', 'mesin', 'jalur', 'short', 'korsleting', 'mati', 'mati total', 'mati cek', 'korosi', 'swap board']):
            damage = 'IC Power Rusak'
        elif match_keywords(sym_text, ['ic cas', 'ic charger', 'ic charging', 'cas rusak', 'hydra']):
            damage = 'IC Cas Rusak'
        elif match_keywords(sym_text, ['baseband', 'ic rf', 'ic signal', 'ic wtr', 'wtr', 'ic rf/baseband']):
            damage = 'IC WTR Rusak'
        elif match_keywords(sym_text, ['antena sinyal', 'antena signal', 'fleksibel antena', 'sinyal', 'signal']):
            damage = 'Antena Signal Rusak'
        elif match_keywords(sym_text, ['kamera', 'lens', 'flash', 'kamera depan', 'kamera belakang']):
            damage = 'Kamera Rusak'
        elif match_keywords(sym_text, ['mikrofon', 'mic', 'microfon']):
            damage = 'Mikrofon Rusak'
        elif match_keywords(sym_text, ['konektor cas', 'port cas', 'fleksibel cas', 'charger kotor', 'cas', 'konektor']):
            damage = 'Port Pengisian Rusak'
        elif match_keywords(sym_text, ['ic audio', 'audio', 'audio rusak']):
            damage = 'IC Audio Rusak'
        elif match_keywords(sym_text, ['backdoor', 'kaca belakang', 'punggung', 'housing', 'casing', 'bodi']):
            damage = 'Backdoor Rusak'
        elif match_keywords(sym_text, ['tombol', 'power', 'volume', 'haptic', 'haptik', 'switch', 'silent', 'home', 'on off', 'on/off']):
            damage = 'Tombol Rusak'

    if damage is None:
        return None
        
    return pd.Series([row['Tipe HP'], signal, baterai, lcd, wifi, touch, tegangan, kamera, konektor, speaker, backdoor, tombol, sym_text.strip(), damage])

mapped_records = []
unmapped_records = []

for idx, row in df_raw.iterrows():
    res = map_row(row)
    if res is not None:
        mapped_records.append(res)
    else:
        unmapped_records.append(row)

df_mapped = pd.DataFrame(mapped_records)
df_mapped.columns = ['Tipe HP', 'Signal', 'Baterai', 'LCD', 'Wifi', 'Touchscreen', 'Tegangan', 'Kamera', 'Konektor Cas', 'Speaker', 'Backdoor', 'Tombol', 'Raw_Text', 'Kerusakan']
df_mapped.head(10)

### 4. Pembagian Dataset (Data Splitting)


In [ ]:
def stratified_split(df, target_col, test_size=0.2, random_state=42):
    counts = df[target_col].value_counts()
    rare_classes = counts[counts < 2].index.tolist()
    
    df_rare = df[df[target_col].isin(rare_classes)]
    df_normal = df[~df[target_col].isin(rare_classes)]
    
    df_train_normal, df_test_normal = train_test_split(
        df_normal, 
        test_size=test_size, 
        random_state=random_state, 
        stratify=df_normal[target_col]
    )
    
    df_train = pd.concat([df_train_normal, df_rare], ignore_index=True)
    df_test = df_test_normal.reset_index(drop=True)
    
    df_train = df_train.sample(frac=1, random_state=random_state).reset_index(drop=True)
    return df_train, df_test

df_train, df_test = stratified_split(df_mapped, 'Kerusakan', test_size=0.2, random_state=42)

print(f"Jumlah Data Latih (80%): {df_train.shape[0]} baris")
print(f"Jumlah Data Uji (20%): {df_test.shape[0]} baris")

### 5. Ekspor Dataset dengan Stempel Waktu (Dynamic Export)

In [ ]:
import datetime

latih_dir = os.path.join(data_dir, "dataLatih")
uji_dir = os.path.join(data_dir, "dataUji")
os.makedirs(latih_dir, exist_ok=True)
os.makedirs(uji_dir, exist_ok=True)

# Mendapatkan waktu saat ini
timestamp = datetime.datetime.now().strftime("%Y-%m-%d-%H-%M")
latih_filename = f"dataLatih-{timestamp}.csv"
uji_filename = f"dataUji-{timestamp}.csv"

# Simpan ke folder masing-masing dengan timestamp
df_train.to_csv(os.path.join(latih_dir, latih_filename), index=False)
df_test.to_csv(os.path.join(uji_dir, uji_filename), index=False)

print(f"Berhasil mengekspor:\n- {os.path.join(latih_dir, latih_filename)}\n- {os.path.join(uji_dir, uji_filename)}")

### 6. Analisis Distribusi Kelas Kerusakan

In [ ]:
plt.figure(figsize=(12, 6))
sns.countplot(y='Kerusakan', data=df_train, order=df_train['Kerusakan'].value_counts().index, hue='Kerusakan', palette='viridis', legend=False)
plt.title('Distribusi Jenis Kerusakan iPhone (Data Latih)')
plt.xlabel('Jumlah Kasus')
plt.ylabel('Kategori Kerusakan')
plt.tight_layout()
plt.show()

### 7. Pelatihan Model Baseline ML (TF-IDF + Random Forest)

In [ ]:
vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=2)
X_train_vec = vectorizer.fit_transform(df_train['Raw_Text'])
X_test_vec = vectorizer.transform(df_test['Raw_Text'])

y_train = df_train['Kerusakan']
y_test = df_test['Kerusakan']

clf = RandomForestClassifier(class_weight='balanced', random_state=42)
clf.fit(X_train_vec, y_train)

y_pred = clf.predict(X_test_vec)

print(f"Accuracy Score: {clf.score(X_test_vec, y_test) * 100:.2f}%")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

### 8. Monitoring Data Tidak Terpetakan (Unmapped Records)

In [ ]:
df_unmapped = pd.DataFrame(unmapped_records)
print(f"Jumlah data tidak terpetakan: {len(df_unmapped)}")
if len(df_unmapped) > 0:
    print("\nSampel data tidak terpetakan (Tipe HP, Keluhan, Tindakan):")
    display_cols = [c for c in ['Tipe HP', 'Symptom_Text', 'Action_Text'] if c in df_unmapped.columns]
    print(df_unmapped[display_cols].head(15))
else:
    print("Semua data berhasil terpetakan dengan sempurna!")